# Enterprise RAG Document QA System

## Project Overview

This project implements a Retrieval-Augmented Generation (RAG) pipeline for answering questions from research PDF documents. It combines semantic search with Large Language Models (LLMs) to retrieve the most relevant document content and generate accurate, context-aware responses.

Instead of relying solely on the knowledge of an LLM, this system first retrieves the most relevant document chunks from a vector database and then uses that context to generate reliable answers.

---

## Project Features

- Load and process multiple PDF research documents
- Split documents into overlapping semantic chunks
- Generate dense vector embeddings using Sentence Transformers
- Store embeddings in ChromaDB for efficient semantic search
- Retrieve the most relevant document chunks
- Generate context-aware responses using a Large Language Model (OpenAI GPT)

---

## Tech Stack

- Python
- LangChain
- Sentence Transformers
- ChromaDB
- OpenAI API
- NumPy

---

## Project Workflow

1. Document Ingestion
2. Text Chunking
3. Embedding Generation
4. Vector Database Creation
5. Semantic Retrieval
6. LLM Response Generation

---

# 1. Document Ingestion

## Objective

The first step of the pipeline is to load research PDF documents from the local directory. Each page is converted into a LangChain `Document` object, which serves as the foundation for text chunking, embedding generation, and semantic retrieval.

This preprocessing step ensures that the documents are represented in a structured format suitable for downstream Retrieval-Augmented Generation (RAG) tasks.

## Import Required Libraries

The following libraries are required to load PDF documents and build the document ingestion pipeline for the RAG system.

In [ ]:
# Import required libraries for loading PDF documents

# Standard Library
import os

# LangChain
from langchain_community.document_loaders.pdf import PyPDFLoader

# 2. PDF Loading

## Objective

This function scans the specified directory for PDF files and loads each document using LangChain's `PyPDFLoader`.

Each page of every PDF is converted into a LangChain `Document` object while preserving useful metadata such as the source file and page number. The resulting collection of documents serves as the input for the text chunking stage.

In [ ]:
def load_all_pdfs():
    folder_path = "pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # Construct the full path to the PDF file
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print(f"Total PDF files loaded: {num_docs}")
    print(f"Total document pages: {len(all_docs)}")
    return all_docs

### Execute the Ingestion Pipeline

The following step loads all research PDFs from the pdfs directory and combines them into a single collection of LangChain Document objects. These documents are then prepared for chunking, embedding generation, and semantic retrieval.

In [ ]:
all_pdf_documents = load_all_pdfs()

# 3. Text Chunking

Large documents are split into smaller overlapping chunks to preserve contextual information while improving semantic search performance. Overlapping chunks help maintain continuity between adjacent sections during retrieval.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    # Configure the text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    # Split LangChain documents into overlapping chunks
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [ ]:
chunked_documents = split_docs(all_pdf_documents)

In [ ]:
print(f"Total chunks created: {len(chunked_documents)}")

# 4. Generate Semantic Embeddings

Each document chunk is converted into a dense vector representation using the Sentence Transformers model (all-MiniLM-L6-v2). These embeddings capture semantic meaning and enable similarity-based retrieval.

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print(f"Loading embedding model: {self.model_name}")
        
        self.model = SentenceTransformer(self.model_name)
        print(
            f"Embedding dimension: {self.model.get_embedding_dimension()}")

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [ ]:
embedding_manager = EmbeddingManager()

# 5. Vector Store

## Objective

This step stores the generated document embeddings in a persistent ChromaDB vector database. Each document chunk is associated with its corresponding embedding, enabling efficient semantic search during the retrieval stage.

Using a vector store allows the RAG system to quickly identify the most relevant document chunks for a user's query without scanning the entire document collection.

In [ ]:
import chromadb
import uuid

In [ ]:
# Create and manage the ChromaDB vector store
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # Initialize the persistent ChromaDB client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # Create or load the document collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("Initialized the vector store with collection:", self.collection_name)
        print("Documents in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

## Execute the Vector Store Pipeline

The following step initializes the ChromaDB vector store and creates the document collection if it does not already exist. The vector database is now ready to store document embeddings for semantic retrieval.

In [ ]:
vector_store = VectorStoreManager()

In [ ]:
# Convert document chunks into embeddings and store them in the vector database

document_texts = [doc.page_content for doc in chunked_documents]

embeddings = embedding_manager.generate_embeddings(document_texts)

vector_store.add_documents(chunked_documents, embeddings)

# 6. Semantic Retrieval

## Objective

This step converts the user's query into an embedding and searches the ChromaDB vector store for the most semantically similar document chunks. The retrieved results provide the contextual information required by the Large Language Model (LLM) to generate accurate and relevant responses.

Semantic retrieval enables the RAG system to answer questions based on the uploaded documents rather than relying solely on the model's pre-trained knowledge.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Retrieve the most relevant document chunks from the vector database
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # Convert the user query into an embedding vector
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # Perform semantic similarity search in the vector store
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # Calculate similarity scores and prepare the retrieved documents
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

## Execute the Retrieval Pipeline

The following step initializes the retrieval engine and searches the vector database for the document chunks that are most semantically similar to the user's query. The retrieved results are returned with their metadata and similarity scores for use during response generation.

In [ ]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [ ]:
query = "What is the Transformer encoder-decoder architecture?"

results = rag_retriever.retrieve(query)

print(f"Retrieved {len(results)} relevant document(s).\n")

for i, result in enumerate(results, start=1):
    print(f"Rank: {i}")
    print(f"Similarity Score: {result['similarity_score']:.4f}")
    print(f"Source: {result['metadata'].get('source', 'Unknown')}")
    print(f"Document Preview:")
    print(result["document"][:300] + "...")
    print("-" * 80)

# 7. Large Language Model (LLM) Integration

## Objective

This stage integrates a Large Language Model (LLM) into the Retrieval-Augmented Generation (RAG) pipeline. The retrieved document chunks are provided as context to the language model, enabling it to generate accurate, context-aware answers instead of relying solely on its pre-trained knowledge.

### API Configuration

This notebook requires an OpenAI API key.
Set the `OPENAI_API_KEY` environment variable before running the notebook.

### OpenAI GPT-5.6

The following model is used to generate natural language responses from the retrieved document context. The retrieved chunks are combined with the user's query and passed to GPT to produce a final answer.

In [ ]:
import os

API_KEY_OPENAI = os.getenv("OPENAI_API_KEY")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.6",
    temperature=0.1,
    max_tokens=1024
)

## Example Query

The following example demonstrates the complete Retrieval-Augmented Generation (RAG) workflow. A user query is converted into an embedding, the most relevant document chunks are retrieved from the vector database, and the retrieved context is provided to the OpenAI GPT model to generate a final response.

In [ ]:
# Generate the final answer using Retrieval-Augmented Generation (RAG)
def generate_output(query, retriever, llm, top_k=3):

    # Retrieve the most relevant document chunks
    results = retriever.retrieve(query, top_k)

    # Combine retrieved document chunks into a single context
    context = "\n".join(doc["document"] for doc in results) if results else ""

    if not context:
        return "No relevant context found for the given query."

    # Create the prompt for the language model
    prompt = f"""
Use only the following context to answer the user's question.
If the answer is not available in the context, say "I could not find the answer in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    # Generate the final response
    response = llm.invoke(prompt)

    return response.content

In [ ]:
answer = generate_output(
    "What is the Transformer encoder-decoder architecture?",
    rag_retriever,
    llm
)

In [ ]:
# Display the generated answer
print(answer)

## 8. Future Enhancements

Possible extensions for this project include:

- Support for multiple LLM providers (OpenAI, Groq, Anthropic)
- Hybrid search
- Conversational memory
- Streaming responses
- RAG evaluation metrics